In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import pandas_ta as ta





# Load and prepare data
asset = "ethereum"

# Load Chainlink candle data
df_1 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/candleData/{asset}_candles.csv")
df_1['date'] = pd.to_datetime(df_1['date'])
df_1.set_index('date', inplace=True)

# Load Chainlink asset data
df_2 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/assetData/{asset}.csv")
df_2['date'] = pd.to_datetime(df_2['date'])
df_2.set_index('date', inplace=True)

# Load Bitcoin candle data
btc_df = pd.read_csv("/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv")
btc_df['date'] = pd.to_datetime(btc_df['date'])
btc_df.set_index('date', inplace=True)

# Merge data
data = pd.merge(df_1, df_2[['total_volume', 'market_cap']], on='date', how='inner')
data.rename(columns={'total_volume': 'Volume', 'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close'}, inplace=True)
data.index.name = 'Date'


if asset != "bitcoin":
    data['close_btc'] = (data['Close'] / btc_df['close']) * 100
    data.dropna(inplace=True)

# Optional: Filter data (uncomment if needed)
#data = data[data.index > '2018-01-01']

data.tail()

In [ ]:
# Calculate features with no lookahead bias (already using min_periods=1)
data['log_returns'] = np.log(data['Close'] / data['Close'].shift(1))
data['log_btc_returns'] = np.log(data['close_btc'] / data['close_btc'].shift(1))
data['MA_30'] = data['Close'].rolling(window=30, min_periods=1).mean()
data['MA_365'] = data['Close'].rolling(window=365, min_periods=1).mean()
data['RSI14'] = ta.rsi(data['Close'], window=14, min_periods=1)
data['CMO'] = ta.cmo(data['Close'], length=14, min_periods=1)
data['ATR'] = ta.atr(data['High'], data['Low'], data['Close'], window=14, min_periods=1)
data['MA_diff'] = data['MA_30'] - data['MA_365']

# Drop rows where any feature is NaN (ensures no incomplete windows)
data.dropna(inplace=True)

# Verify the tail to ensure data looks correct
print(data.tail())

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Define features for clustering (remove CMO due to redundancy with RSI14)
features_for_clustering = ['log_returns', 'MA_diff', 'RSI14', 'log_btc_returns', 'ATR']

# Split data into train (before 2025) and test (2025 onwards)
train_data = data[data.index < '2025-01-01'][features_for_clustering].dropna()
test_data = data[data.index >= '2025-01-01'][features_for_clustering].dropna()

# Fit RobustScaler and KMeans on training data
scaler = RobustScaler()
train_scaled = scaler.fit_transform(train_data)

kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(train_scaled)

# Apply to test data (transform, not refit)
test_scaled = scaler.transform(test_data)

# Assign clusters to both train and test data
data.loc[train_data.index, 'Cluster'] = kmeans.labels_
data.loc[test_data.index, 'Cluster'] = kmeans.predict(test_scaled)

# Verify the Cluster assignments
print(data[['Close', 'Cluster']].tail())

# Evaluate clustering quality with Silhouette Score
print("Silhouette Score on Train Set:", silhouette_score(train_scaled, kmeans.labels_))
print("Silhouette Score on Test Set:", silhouette_score(test_scaled, kmeans.predict(test_scaled)))

In [ ]:
import plotly.graph_objects as go

# Function to predict clusters for new incoming data using fitted models
def predict_new_clusters(new_data, scaler, kmeans, features_for_clustering):
    # Ensure new_data has the same features and is a DataFrame
    new_data = new_data[features_for_clustering].dropna()
    if not new_data.empty:
        new_scaled = scaler.transform(new_data)
        new_clusters = kmeans.predict(new_scaled)
        return new_clusters
    return None

# Example: Simulate new incoming data (you’d replace this with actual new ETH data)
# For now, we’ll use the test data as an example of "new data"
new_data_example = test_data.copy()

# Predict clusters for the example new data (this is redundant but shows the process)
new_clusters = predict_new_clusters(new_data_example, scaler, kmeans, features_for_clustering)

# Assign new clusters to the DataFrame (if any)
if new_clusters is not None:
    data.loc[new_data_example.index, 'Cluster'] = new_clusters

# Create a Plotly chart to visualize ETH price with cluster classifications
fig = go.Figure()

# Add the continuous price line
fig.add_trace(
    go.Scatter(
        x=data.index,
        y=data['Close'],
        mode='lines',
        name='Price',
        line=dict(color='gray', width=1)
    )
)

# Add scatter points colored by cluster (only for dates with assigned clusters)
colors = ['darkred', 'darkgreen', 'goldenrod']  # Cluster 0, 1, 2
for i in range(3):
    cluster_data = data[data['Cluster'] == i].dropna(subset=['Cluster'])
    if not cluster_data.empty:
        fig.add_trace(
            go.Scatter(
                x=cluster_data.index,
                y=cluster_data['Close'],
                mode='markers',
                name=f'Cluster {i}',
                marker=dict(
                    color=colors[i],
                    size=4
                )
            )
        )

# Update layout
fig.update_layout(
    title='ETH Price with Cluster Classifications (Train + Test)',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    hovermode='x unified',
    width=1200,
    height=600
)

fig.show()



# Optional: Print cluster stats for interpretation
cluster_stats = pd.DataFrame()
for i in range(3):
    cluster_data = data[data['Cluster'] == i]['log_returns'].dropna()
    stats = {
        'Mean Returns': cluster_data.mean(),
        'Std Dev': cluster_data.std(),
        'Min': cluster_data.min(),
        'Max': cluster_data.max(),
        'Skewness': cluster_data.skew(),
        'Kurtosis': cluster_data.kurtosis(),
        'Size': len(cluster_data)
    }
    cluster_stats[f'Cluster {i}'] = pd.Series(stats)

print("\nCluster Statistics:")
print(cluster_stats.round(3))